# AgentTrace — Callback handler (cached-agent servers)

`AgentTraceCallbackHandler` is a LangChain `AsyncCallbackHandler` you attach **per request** via `config={"callbacks": [handler]}`. It's the right tool when your server **compiles the agent once and reuses it** across requests (a middleware, baked into the graph at build time, would span multiple runs).

One top-level handler fires for the main agent **and every deepagents sub-agent** (LangGraph propagates callbacks down the ambient config), capturing: LLM input/output/tokens, tool call/result, sub-agent handoffs, and errors — each attributed to the main agent or the emitting sub-agent.

```bash
pip install agenttrace-langchain
```

> Runs **fully offline**: fake model + a mocked HTTP transport that captures the emitted events. Real-server config is at the end.

In [ ]:
import json
import httpx
from agenttrace_langchain import AsyncAgentTraceClient, AgentTraceCallbackHandler


def capture_client():
    """An AsyncAgentTraceClient whose POSTs are captured in-memory (no network)."""
    bodies = []

    def handler(request):
        body = json.loads(request.content.decode())
        bodies.append(body)
        run = {'runId': 'run_demo'}
        return httpx.Response(201, json={**run, 'event': {}} if body.get('runId') else run)

    client = AsyncAgentTraceClient(
        url='http://demo/api/events', api_key='atr_demo',
        client=httpx.AsyncClient(transport=httpx.MockTransport(handler)),
    )
    return client, bodies


def show(bodies):
    for b in bodies:
        if 'type' in b:
            print(f"{b['type']:13} {b['source']:14} -> {b['target']:14} {b.get('label','')}")
        elif 'endRun' in b:
            print(f"--- run ended: {b['endRun']} ---")

## A delegating agent

We build a real `create_deep_agent` with one sub-agent (`researcher`) and a fake model scripted to **delegate** via the `task` tool, then answer.

In [ ]:
from langchain_core.messages import AIMessage
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from deepagents import create_deep_agent


class FakeToolModel(FakeMessagesListChatModel):
    def bind_tools(self, tools, **kwargs):
        return self


def build_agent():
    # Order of consumption: lead delegates -> sub-agent answers -> lead answers.
    model = FakeToolModel(responses=[
        AIMessage(content="", tool_calls=[{"name": "task",
            "args": {"subagent_type": "researcher", "description": "find X"}, "id": "d1"}]),
        AIMessage(content="I found X."),
        AIMessage(content="Done: X."),
    ])
    return create_deep_agent(
        model=model, tools=[],
        subagents=[{"name": "researcher", "description": "researches things",
                    "system_prompt": "You research."}],
    )

## Trace a run

Create a handler per request, attach it to `config["callbacks"]`, then feed the app-only lifecycle a callback can't derive: `on_user_message` and, at the end, `finish` (the assembled answer) + `aclose` (bounded drain).

In [ ]:
client, bodies = capture_client()
handler = AgentTraceCallbackHandler('chat run', client=client)
handler.on_user_message('research X')

agent = build_agent()
config = {'callbacks': [handler]}
async for _ in agent.astream_events(
    {'messages': [{'role': 'user', 'content': 'research X'}]}, version='v2', config=config,
):
    pass  # your own UI dispatch goes here — tracing is entirely in the callback

await handler.finish('completed', answer='Done: X.')
await handler.aclose()

show(bodies)

Note the **sub-agent attribution**: the `task` call becomes a `handoff` (Orchestrator → researcher), and the sub-agent's own `llm_call` is attributed to `researcher`, not the orchestrator — even though LangGraph labels every LLM node structurally `"model"`. The handler recovers the sub-agent from the `task` call's `checkpoint_ns`.

## PII anonymization

Pass an `anonymizer` (any `Callable[[Any], Any]`) once; it runs on **every** event payload right before send — instead of masking at each call site. It's never fatal: a raising anonymizer logs a warning and the payload passes through. Pair it with, e.g., `langsmith.anonymizer.create_anonymizer(rules)` for real redaction.

In [ ]:
# Toy anonymizer: replace every payload with a masked marker to prove it runs everywhere.
client, bodies = capture_client()
handler = AgentTraceCallbackHandler(
    'redacted run', client=client, anonymizer=lambda payload: {'masked': True},
)
handler.on_user_message('patient Jane Doe, SSN 123-45-6789')

agent = build_agent()
async for _ in agent.astream_events(
    {'messages': [{'role': 'user', 'content': 'research X'}]}, version='v2',
    config={'callbacks': [handler]},
):
    pass
await handler.finish('completed', answer='sensitive result')
await handler.aclose()

for b in bodies:
    if 'payload' in b:
        print(b['type'], '->', b['payload'])

## Localized diagram labels (`phrases`)

Override any of the default English labels (merged over `async_run.DEFAULT_PHRASES`). `{name}` / `{target}` are filled with `str.format`.

In [ ]:
client, bodies = capture_client()
handler = AgentTraceCallbackHandler(
    'run FR', client=client,
    phrases={
        'delegate': 'délégation → {target}',
        'subagent_done': '{name} → retour',
        'final_answer': 'réponse finale',
    },
)
handler.on_user_message('recherche X')
agent = build_agent()
async for _ in agent.astream_events(
    {'messages': [{'role': 'user', 'content': 'recherche X'}]}, version='v2',
    config={'callbacks': [handler]},
):
    pass
await handler.finish('completed', answer='Terminé : X.')
await handler.aclose()

show(bodies)

## Lifecycle & real usage

The callback captures LLM/tool/sub-agent events on its own. The app supplies only what a callback can't observe:

- `handler.on_user_message(text)` — the opening User → Orchestrator arrow.
- `handler.approval_required(info)` — a **HITL pause** (read from graph state, not a callback event).
- `await handler.finish(status, answer=...)` — emit the assembled final answer and mark the run ended.
- `await handler.aclose()` — drain the queue (bounded). Call it **after** you've sent your own `done` to the client so tracing never adds latency.

Against a real deployment, build the client once and reuse it across runs:

```python
from agenttrace_langchain import AsyncAgentTraceClient, AgentTraceCallbackHandler

client = AsyncAgentTraceClient(api_key='atr_...')  # or set AGENTTRACE_URL / AGENTTRACE_KEY

# per request:
handler = AgentTraceCallbackHandler(
    'chat run', client=client,
    anonymizer=my_scrubber,             # optional
    tool_server=my_mcp_routing_fn,      # optional: label which backend served a tool
)
handler.on_user_message(user_text)
config = {'configurable': {'thread_id': tid}, 'callbacks': [handler]}
async for event in agent.astream_events(payload, version='v2', config=config):
    ...                                  # your UI dispatch, untouched by tracing
await handler.finish('completed', answer=final_report)
await handler.aclose()
```

Real integrations using this exact pattern: `siigma-chat` and `statistic-ai-agent` (a per-`session_id` handler attached to each `astream_events` call). Launch the dashboard with `pip install deepagents-trace && agenttrace ui`.